# EEGMMIDB quick reader

This notebook loads a local file from the PhysioNet EEG Motor Movement/Imagery dataset (`eegmmidb`).

- File format: EDF+ (`.edf`), 64 EEG channels, 160 Hz
- Annotation codes in each run: `T0` (rest), `T1`, `T2`
- `T1`/`T2` meanings depend on run number, so we decode them using the run-task map.

In [46]:
from pathlib import Path
from collections import Counter

import mne

from mne.datasets import eegbci

# Choose a file to load
SUBJECT = 1
RUN = 6  # Try 4, 6, 10, 14 for motor imagery runs

# Candidate roots for your local download
CANDIDATE_ROOTS = [
    Path("MNE-eegbci-data/files/eegmmidb/1.0.0"),
    Path("MNE-eegbci-data/eegmmidb/1.0.0"),
    Path("files/eegmmidb/1.0.0"),
]

def resolve_data_root(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    checked = "\n".join(f"- {c.resolve()}" for c in candidates)
    raise FileNotFoundError(
        "Could not find eegmmidb root. Checked:\n" + checked
    )

data_root = resolve_data_root(CANDIDATE_ROOTS)

edf_path = data_root / f"S{SUBJECT:03d}" / f"S{SUBJECT:03d}R{RUN:02d}.edf"

if not edf_path.exists():
    raise FileNotFoundError(f"Missing file: {edf_path}")

print("Data root:", data_root)
print("EDF file:", edf_path)


Data root: C:\Users\Kades\Documents\MAU\AML GP\BCI\MNE-eegbci-data\files\eegmmidb\1.0.0
EDF file: C:\Users\Kades\Documents\MAU\AML GP\BCI\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R06.edf


In [47]:
raw = mne.io.read_raw_edf(edf_path, infer_types=True, preload=False, verbose="ERROR")

eegbci.standardize(raw)

raw.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")

print(raw)
print(f"Sampling frequency: {raw.info['sfreq']} Hz")
print(f"Number of channels: {len(raw.ch_names)}")
print("First 8 channels:", raw.ch_names[:8])


<RawEDF | S001R06.edf, 64 x 20000 (125.0 s), ~75 KiB, data not loaded>
Sampling frequency: 160.0 Hz
Number of channels: 64
First 8 channels: ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'C5']


In [48]:
events, event_id = mne.events_from_annotations(raw, verbose="ERROR")

id_to_code = {v: k for k, v in event_id.items()}
code_counts = Counter(id_to_code[e[-1]] for e in events)

RUN_TASK = {
    1: "Baseline, eyes open",
    2: "Baseline, eyes closed",
    3: "Motor execution: left vs right hand",
    4: "Motor imagery: left vs right hand",
    5: "Motor execution: hands vs feet",
    6: "Motor imagery: hands vs feet",
    7: "Motor execution: left vs right hand",
    8: "Motor imagery: left vs right hand",
    9: "Motor execution: hands vs feet",
    10: "Motor imagery: hands vs feet",
    11: "Motor execution: left vs right hand",
    12: "Motor imagery: left vs right hand",
    13: "Motor execution: hands vs feet",
    14: "Motor imagery: hands vs feet",
}

if RUN in {3, 4, 7, 8, 11, 12}:
    semantic_map = {"T0": "rest", "T1": "left fist", "T2": "right fist"}
elif RUN in {5, 6, 9, 10, 13, 14}:
    semantic_map = {"T0": "rest", "T1": "both fists", "T2": "both feet"}
else:
    semantic_map = {"T0": "rest"}

semantic_counts = {semantic_map.get(code, code): count for code, count in code_counts.items()}

print("Run:", RUN, "->", RUN_TASK.get(RUN, "Unknown run"))
print("event_id from MNE:", event_id)
print("Counts by raw code:", dict(code_counts))
print("Counts by semantic label:", semantic_counts)
print("First 5 events [sample, 0, id]:")
print(events[:5])


Run: 6 -> Motor imagery: hands vs feet
event_id from MNE: {np.str_('T0'): 1, np.str_('T1'): 2, np.str_('T2'): 3}
Counts by raw code: {np.str_('T0'): 15, np.str_('T2'): 8, np.str_('T1'): 7}
Counts by semantic label: {'rest': 15, 'both feet': 8, 'both fists': 7}
First 5 events [sample, 0, id]:
[[   0    0    1]
 [ 672    0    3]
 [1328    0    1]
 [2000    0    2]
 [2656    0    1]]


## Option 1: Feasibility test (Executed vs Imagined) with CSP + LDA

This section evaluates whether executed and imagined movement can be separated for the **same task family**:
- left/right fist: run pairs (3,4), (7,8), (11,12)
- both fists/feet: run pairs (5,6), (9,10), (13,14)

Labels are binary: `0 = executed`, `1 = imagined`.


In [49]:
# Install once if needed:
# %pip install scikit-learn

import time
import numpy as np
import mne
from mne.datasets import eegbci

try:
    from mne.decoding import CSP
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.pipeline import Pipeline
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "scikit-learn is required for Option 1. Run: %pip install scikit-learn"
    ) from exc

# Keep logs compact during repeated CV fits
mne.set_log_level("WARNING")

# Evaluation settings
SUBJECTS = list(range(1, 110))  # many subjects
TASK_FAMILIES = ["left_right", "hands_feet"]
TMIN, TMAX = 0.5, 2.5  # seconds after cue
L_FREQ, H_FREQ = 8.0, 30.0
N_COMPONENTS = 6
PROGRESS_EVERY = 10  # print progress every N subjects

RUN_PAIR_FAMILIES = {
    "left_right": [(3, 4), (7, 8), (11, 12)],
    "hands_feet": [(5, 6), (9, 10), (13, 14)],
}

def load_task_epochs(subject, run, data_root, tmin, tmax, l_freq, h_freq):
    edf_path = data_root / f"S{subject:03d}" / f"S{subject:03d}R{run:02d}.edf"
    if not edf_path.exists():
        return None

    raw = mne.io.read_raw_edf(edf_path, infer_types=True, preload=True, verbose="ERROR")
    eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")
    raw.pick("eeg")
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose="ERROR")

    events, event_id = mne.events_from_annotations(
        raw, event_id={"T1": 1, "T2": 2}, verbose="ERROR"
    )
    if len(events) == 0:
        return None

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,
        preload=True,
        verbose="ERROR",
    )
    return epochs

def build_exec_vs_imag_dataset(subject, task_family):
    pairs = RUN_PAIR_FAMILIES[task_family]
    X_all = []
    y_all = []
    groups_all = []

    for run_exec, run_imag in pairs:
        ep_exec = load_task_epochs(subject, run_exec, data_root, TMIN, TMAX, L_FREQ, H_FREQ)
        ep_imag = load_task_epochs(subject, run_imag, data_root, TMIN, TMAX, L_FREQ, H_FREQ)

        if ep_exec is None or ep_imag is None:
            continue

        X_exec = ep_exec.get_data(copy=False)
        X_imag = ep_imag.get_data(copy=False)

        X_all.append(X_exec)
        y_all.append(np.zeros(len(X_exec), dtype=int))
        groups_all.append(np.full(len(X_exec), run_exec, dtype=int))

        X_all.append(X_imag)
        y_all.append(np.ones(len(X_imag), dtype=int))
        groups_all.append(np.full(len(X_imag), run_imag, dtype=int))

    if not X_all:
        return None, None, None

    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)
    groups = np.concatenate(groups_all, axis=0)
    return X, y, groups

def make_leave_runpair_out_splits(groups, task_family):
    """Create grouped folds: each test fold holds out one exec+imag run pair."""
    pairs = RUN_PAIR_FAMILIES[task_family]
    unique_groups = set(np.unique(groups).tolist())
    splits = []

    for run_exec, run_imag in pairs:
        if run_exec not in unique_groups or run_imag not in unique_groups:
            continue

        test_mask = np.isin(groups, [run_exec, run_imag])
        train_idx = np.where(~test_mask)[0]
        test_idx = np.where(test_mask)[0]

        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        splits.append((train_idx, test_idx))

    return splits

def evaluate_subject_grouped(subject, task_family):
    X, y, groups = build_exec_vs_imag_dataset(subject, task_family)
    if X is None:
        return None

    splits = make_leave_runpair_out_splits(groups, task_family)
    if len(splits) < 2:
        return None

    clf = Pipeline([
        ("csp", CSP(n_components=N_COMPONENTS, reg="ledoit_wolf", log=True, norm_trace=False)),
        ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ])

    fold_scores = []
    for train_idx, test_idx in splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        score = balanced_accuracy_score(y_test, y_pred)
        fold_scores.append(score)

    fold_scores = np.array(fold_scores, dtype=float)
    return {
        "subject": subject,
        "n_trials": int(len(y)),
        "n_exec": int((y == 0).sum()),
        "n_imag": int((y == 1).sum()),
        "n_folds": int(len(fold_scores)),
        "mean_bal_acc": float(np.mean(fold_scores)),
        "std_bal_acc": float(np.std(fold_scores)),
        "scores": fold_scores,
    }



In [50]:
all_family_results = {}

for task_family in TASK_FAMILIES:
    print(f"\n=== Evaluating task family: {task_family} ===")
    family_start = time.perf_counter()
    results = []

    total = len(SUBJECTS)
    for i, sub in enumerate(SUBJECTS, start=1):
        res = evaluate_subject_grouped(sub, task_family)
        if res is not None:
            results.append(res)

        if i % PROGRESS_EVERY == 0 or i == total:
            elapsed = time.perf_counter() - family_start
            print(
                f"[{task_family}] processed {i}/{total} subjects | "
                f"valid={len(results)} | elapsed={elapsed:.1f}s"
            )

    all_family_results[task_family] = results

# Detailed per-subject output (optional):
# for task_family, results in all_family_results.items():
#     for r in results:
#         print(
#             f"{task_family} S{r['subject']:03d} | folds={r['n_folds']} | "
#             f"balanced_acc={r['mean_bal_acc']:.3f} +/- {r['std_bal_acc']:.3f}"
#         )

print("\n=== Side-by-side summary ===")
print("family      | subjects | mean_bal_acc | std_across_subjects")
print("------------|----------|--------------|--------------------")

for task_family in TASK_FAMILIES:
    results = all_family_results.get(task_family, [])
    if not results:
        print(f"{task_family:11} | {0:8d} | {'n/a':>12} | {'n/a':>18}")
        continue

    subject_means = np.array([r['mean_bal_acc'] for r in results], dtype=float)
    print(
        f"{task_family:11} | {len(results):8d} | "
        f"{subject_means.mean():12.3f} | {subject_means.std():18.3f}"
    )

print("\nChance level is 0.500 for this binary task.")




=== Evaluating task family: left_right ===
[left_right] processed 10/109 subjects | valid=10 | elapsed=15.1s
[left_right] processed 20/109 subjects | valid=20 | elapsed=30.6s
[left_right] processed 30/109 subjects | valid=30 | elapsed=45.1s
[left_right] processed 40/109 subjects | valid=40 | elapsed=60.0s
[left_right] processed 50/109 subjects | valid=50 | elapsed=78.3s
[left_right] processed 60/109 subjects | valid=60 | elapsed=96.9s
[left_right] processed 70/109 subjects | valid=70 | elapsed=114.2s
[left_right] processed 80/109 subjects | valid=80 | elapsed=129.1s
[left_right] processed 90/109 subjects | valid=90 | elapsed=144.0s
[left_right] processed 100/109 subjects | valid=100 | elapsed=161.8s
[left_right] processed 109/109 subjects | valid=109 | elapsed=178.4s

=== Evaluating task family: hands_feet ===
[hands_feet] processed 10/109 subjects | valid=10 | elapsed=16.5s
[hands_feet] processed 20/109 subjects | valid=20 | elapsed=33.8s
[hands_feet] processed 30/109 subjects | vali

## Option 4: Compare all EEG channels vs sensorimotor subset

This cell runs the same grouped-CV evaluation with two channel modes:
- `all`: all EEG channels
- `sensorimotor`: channels around FC/C/CP (motor cortex neighborhood)

Use this to test whether restricting channels improves execution-vs-imagery classification.


In [51]:
# Channel subset comparison (uses same grouped run-pair CV idea)

# You can change these if needed:
COMPARE_TASK_FAMILIES = TASK_FAMILIES
COMPARE_SUBJECTS = SUBJECTS
COMPARE_PROGRESS_EVERY = 10

SENSORIMOTOR_CHANNELS = [
    "FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6",
    "C5", "C3", "C1", "Cz", "C2", "C4", "C6",
    "CP5", "CP3", "CP1", "CPz", "CP2", "CP4", "CP6",
]


def load_task_epochs_channel_mode(subject, run, data_root, tmin, tmax, l_freq, h_freq, channel_mode="all"):
    edf_path = data_root / f"S{subject:03d}" / f"S{subject:03d}R{run:02d}.edf"
    if not edf_path.exists():
        return None

    raw = mne.io.read_raw_edf(edf_path, infer_types=True, preload=True, verbose="ERROR")
    eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")
    raw.pick("eeg")

    if channel_mode == "sensorimotor":
        picks = [ch for ch in SENSORIMOTOR_CHANNELS if ch in raw.ch_names]
        if len(picks) < 8:
            return None
        raw.pick(picks)

    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose="ERROR")

    events, event_id = mne.events_from_annotations(
        raw, event_id={"T1": 1, "T2": 2}, verbose="ERROR"
    )
    if len(events) == 0:
        return None

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,
        preload=True,
        verbose="ERROR",
    )
    return epochs


def build_exec_vs_imag_dataset_channel_mode(subject, task_family, channel_mode="all"):
    pairs = RUN_PAIR_FAMILIES[task_family]
    X_all = []
    y_all = []
    groups_all = []

    for run_exec, run_imag in pairs:
        ep_exec = load_task_epochs_channel_mode(
            subject, run_exec, data_root, TMIN, TMAX, L_FREQ, H_FREQ, channel_mode=channel_mode
        )
        ep_imag = load_task_epochs_channel_mode(
            subject, run_imag, data_root, TMIN, TMAX, L_FREQ, H_FREQ, channel_mode=channel_mode
        )

        if ep_exec is None or ep_imag is None:
            continue

        X_exec = ep_exec.get_data(copy=False)
        X_imag = ep_imag.get_data(copy=False)

        X_all.append(X_exec)
        y_all.append(np.zeros(len(X_exec), dtype=int))
        groups_all.append(np.full(len(X_exec), run_exec, dtype=int))

        X_all.append(X_imag)
        y_all.append(np.ones(len(X_imag), dtype=int))
        groups_all.append(np.full(len(X_imag), run_imag, dtype=int))

    if not X_all:
        return None, None, None

    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)
    groups = np.concatenate(groups_all, axis=0)
    return X, y, groups


def evaluate_subject_grouped_channel_mode(subject, task_family, channel_mode="all"):
    X, y, groups = build_exec_vs_imag_dataset_channel_mode(subject, task_family, channel_mode=channel_mode)
    if X is None:
        return None

    splits = make_leave_runpair_out_splits(groups, task_family)
    if len(splits) < 2:
        return None

    clf = Pipeline([
        ("csp", CSP(n_components=N_COMPONENTS, reg="ledoit_wolf", log=True, norm_trace=False)),
        ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ])

    fold_scores = []
    for train_idx, test_idx in splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        score = balanced_accuracy_score(y_test, y_pred)
        fold_scores.append(score)

    fold_scores = np.array(fold_scores, dtype=float)
    return {
        "subject": subject,
        "n_trials": int(len(y)),
        "n_channels": int(X.shape[1]),
        "mean_bal_acc": float(np.mean(fold_scores)),
        "std_bal_acc": float(np.std(fold_scores)),
    }


compare_results = {mode: {} for mode in ["all", "sensorimotor"]}

for mode in ["all", "sensorimotor"]:
    print(f"\n=== Channel mode: {mode} ===")
    mode_start = time.perf_counter()

    for task_family in COMPARE_TASK_FAMILIES:
        results = []
        total = len(COMPARE_SUBJECTS)

        for i, sub in enumerate(COMPARE_SUBJECTS, start=1):
            res = evaluate_subject_grouped_channel_mode(sub, task_family, channel_mode=mode)
            if res is not None:
                results.append(res)

            if i % COMPARE_PROGRESS_EVERY == 0 or i == total:
                elapsed = time.perf_counter() - mode_start
                print(
                    f"[{mode} | {task_family}] {i}/{total} subjects | "
                    f"valid={len(results)} | elapsed={elapsed:.1f}s"
                )

        compare_results[mode][task_family] = results


print("\n=== Side-by-side channel comparison ===")
print("mode         | family      | subjects | mean_bal_acc | std_across_subjects")
print("-------------|-------------|----------|--------------|--------------------")

for mode in ["all", "sensorimotor"]:
    for task_family in COMPARE_TASK_FAMILIES:
        results = compare_results[mode][task_family]
        if not results:
            print(f"{mode:12} | {task_family:11} | {0:8d} | {'n/a':>12} | {'n/a':>18}")
            continue

        subject_means = np.array([r["mean_bal_acc"] for r in results], dtype=float)
        print(
            f"{mode:12} | {task_family:11} | {len(results):8d} | "
            f"{subject_means.mean():12.3f} | {subject_means.std():18.3f}"
        )




=== Channel mode: all ===
[all | left_right] 10/109 subjects | valid=10 | elapsed=15.5s
[all | left_right] 20/109 subjects | valid=20 | elapsed=31.2s
[all | left_right] 30/109 subjects | valid=30 | elapsed=48.1s
[all | left_right] 40/109 subjects | valid=40 | elapsed=66.9s
[all | left_right] 50/109 subjects | valid=50 | elapsed=84.7s
[all | left_right] 60/109 subjects | valid=60 | elapsed=105.3s
[all | left_right] 70/109 subjects | valid=70 | elapsed=123.4s
[all | left_right] 80/109 subjects | valid=80 | elapsed=139.5s
[all | left_right] 90/109 subjects | valid=90 | elapsed=157.1s
[all | left_right] 100/109 subjects | valid=100 | elapsed=173.1s
[all | left_right] 109/109 subjects | valid=109 | elapsed=189.2s
[all | hands_feet] 10/109 subjects | valid=10 | elapsed=207.8s
[all | hands_feet] 20/109 subjects | valid=20 | elapsed=223.4s
[all | hands_feet] 30/109 subjects | valid=30 | elapsed=239.7s
[all | hands_feet] 40/109 subjects | valid=40 | elapsed=257.8s
[all | hands_feet] 50/109 sub